## vast.ai — conda setup (no venv)

Before running:
1. Pull latest code on instance: `git -C /root/NER_translation pull`
2. Sync checkpoint from Drive (vast.ai UI → Cloud Copy):
   - **Drive → Instance**: `BUyen_Qnhu++/src/SeqDiffuSeq/ckpts/en-zh/` → `/root/NER_translation/SeqDiffuSeq/ckpts/en-zh/`
3. Open this notebook, run all cells.

## Phase 1 — Environment Bootstrap

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    # vast.ai: no Drive mount needed.
    # Use vastai copy CLI (from your local machine) to sync before/after training:
    #   Pull checkpoint:  vastai copy drive:/BUyen_Qnhu++/src/SeqDiffuSeq/ckpts/en-zh/ INSTANCE_ID:/root/NER_translation/SeqDiffuSeq/ckpts/en-zh/
    #   Push checkpoint:  vastai copy INSTANCE_ID:/root/NER_translation/SeqDiffuSeq/ckpts/en-zh/ drive:/BUyen_Qnhu++/src/SeqDiffuSeq/ckpts/en-zh/
    print("vast.ai — skipping Drive mount. Use 'vastai copy' to sync checkpoints.")

In [ ]:
import os
import shutil
import subprocess


In [ ]:
import os

# ── CONFIG ───────────────────────────────────────────────────────────────
CONDA_ENV   = "seqdiff"   # conda env name — change if you used a different name
# ─────────────────────────────────────────────────────────────────────────

PLATFORM    = "vastai"
REPO_DIR    = "/root/NER_translation/SeqDiffuSeq"
DRIVE_DATA  = "/root/train_dataset"
VENV_PYTHON = f"/root/miniconda3/envs/{CONDA_ENV}/bin/python"

DST_DATA_DIR = os.path.join(REPO_DIR, "data", "en-zh")
CKPT_DIR     = os.path.join(REPO_DIR, "ckpts", "en-zh")
PRETRAINED   = os.path.join(REPO_DIR, "pretrained", "bart-base")
OUT_DIR      = os.path.join(CKPT_DIR, "inference_out")

print(f"Conda env : {CONDA_ENV}")
print(f"Python    : {VENV_PYTHON}")
print(f"Repo      : {REPO_DIR}")
print(f"Checkpts  : {CKPT_DIR}")

In [2]:
# Cell 0 – GPU Check
# Run this first. If CUDA is False, go to: Runtime > Change runtime type > T4 GPU
import torch

print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU            :", torch.cuda.get_device_name(0))
    print("VRAM           :", round(props.total_memory / 1024**3, 1), "GB")
    print("PyTorch        :", torch.__version__)

else:
    print()
    print("⚠️  No GPU detected.")
    print("   Go to: Runtime > Change runtime type > Hardware accelerator > T4 GPU")
    print("   Save, wait for reconnect, then re-run this cell.")

CUDA available : True
GPU            : Tesla T4
VRAM           : 14.6 GB
PyTorch        : 2.8.0+cu126


In [ ]:
!sudo apt update



Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease     
Hit:4 https://cli.github.com/packages stable InRelease                         
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease               
Hit:7 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease                     
Ign:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease   
Ign:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Ign:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Ign:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Ign:10 https://ppa.launchpadco

In [ ]:
# Create conda env with Python 3.10 (skip if it already exists)
result = subprocess.run(f"conda env list | grep -q '^{CONDA_ENV} '", shell=True)
if result.returncode == 0:
    print(f"Conda env '{CONDA_ENV}' already exists — skipping.")
else:
    subprocess.run(f"conda create -n {CONDA_ENV} python=3.10 -y", shell=True, check=True)
    print(f"Conda env '{CONDA_ENV}' created.")

In [ ]:
subprocess.run([VENV_PYTHON, "-m", "pip", "install", "--upgrade", "pip==23"], check=True)

In [ ]:
subprocess.run([
    VENV_PYTHON, "-m", "pip", "install", "-q",
    "torch==1.11.0+cu113", "torchvision==0.12.0+cu113", "torchaudio==0.11.0",
    "--extra-index-url", "https://download.pytorch.org/whl/cu113"
], check=True)

subprocess.run([
    VENV_PYTHON, "-m", "pip", "install", "-q",
    "bert-score", "blobfile", "datasets",
    "huggingface-hub==0.4.0",
    "mpi4py", "nltk", "numpy", "pandas", "protobuf",
    "rouge-score", "sacrebleu", "sacremoses",
    "scikit-learn", "scipy", "spacy",
    "tokenizers", "torchmetrics", "tqdm",
    "transformers==4.18.0",
], check=True)

print("Dependencies installed.")
subprocess.run([VENV_PYTHON, "-c", "import torch; print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())"])

## Phase 2 — Data Setup

In [51]:
# Paths are set in the CONFIG cell above.
assert "REPO_DIR" in dir(), "Run the CONFIG cell first."


In [52]:

required_files = [
    "train_clean.en", "train_clean.zh",
    "valid_clean.en", "valid_clean.zh",
    "test_clean.en",  "test_clean.zh",
]

all_ok = True
for fname in required_files:
    path = os.path.join(DRIVE_DATA, fname)
    if os.path.exists(path):
        n = sum(1 for _ in open(path, encoding='utf-8'))
        print(f"  \u2713  {fname:30s}  {n:>8,} lines")
    else:
        print(f"  \u2717  {fname:30s}  NOT FOUND")
        all_ok = False


  ✓  train_clean.en                   233,842 lines
  ✓  train_clean.zh                   233,842 lines
  ✓  valid_clean.en                    29,236 lines
  ✓  valid_clean.zh                    29,236 lines
  ✓  test_clean.en                     29,253 lines
  ✓  test_clean.zh                     29,253 lines


In [53]:
os.makedirs(DST_DATA_DIR, exist_ok=True)

# Copy cleaned files; destination keeps original names so training scripts need no changes.
file_mapping = {
    "train_clean.en": "train.en",
    "train_clean.zh": "train.zh",
    "valid_clean.en": "valid.en",
    "valid_clean.zh": "valid.zh",
    "test_clean.en":  "test.en",
    "test_clean.zh":  "test.zh",
}

print(f"Copying to: {DST_DATA_DIR}\n")
for src_name, dst_name in file_mapping.items():
    src = os.path.join(DRIVE_DATA, src_name)
    dst = os.path.join(DST_DATA_DIR, dst_name)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing: {src} — run clean_dataset.py first.")
    shutil.copy2(src, dst)
    n = sum(1 for _ in open(dst, encoding='utf-8'))
    print(f"  Copied  {src_name} -> {dst_name:20s}  ({n:,} lines)")

print(f"\nData ready. Repo data dir:")
print(f"   {sorted(os.listdir(DST_DATA_DIR))}")


Copying to: /content/drive/MyDrive/BUyen_Qnhu++/src/SeqDiffuSeq/data/en-zh

  Copied  train_clean.en -> train.en              (233,842 lines)
  Copied  train_clean.zh -> train.zh              (233,842 lines)
  Copied  valid_clean.en -> valid.en              (29,236 lines)
  Copied  valid_clean.zh -> valid.zh              (29,236 lines)
  Copied  test_clean.en -> test.en               (29,253 lines)
  Copied  test_clean.zh -> test.zh               (29,253 lines)

Data ready. Repo data dir:
   ['test.en', 'test.zh', 'train.en', 'train.zh', 'valid.en', 'valid.zh']


## Phase 3 — Tokenizer Training

In [ ]:
import shutil, time
from tokenizers import ByteLevelBPETokenizer

LOCAL_TOK_DIR = "/tmp/tok_tmp"
os.makedirs(LOCAL_TOK_DIR, exist_ok=True)

# Copy train files to local SSD if not already there
for fname in ["train.en", "train.zh"]:
    dst = os.path.join(LOCAL_TOK_DIR, fname)
    if not os.path.exists(dst):
        print(f"Copying {fname} to local storage…")
        shutil.copy2(os.path.join(DST_DATA_DIR, fname), dst)

data_files = sorted([os.path.join(LOCAL_TOK_DIR, f)
                     for f in os.listdir(LOCAL_TOK_DIR) if "train" in f])
print(f"Training on: {data_files}")

t0 = time.time()
tokenizer = ByteLevelBPETokenizer()
tokenizer.train(
    files=data_files,
    vocab_size=32005,
    min_frequency=2,
    special_tokens=["<s>", "<pad>", "</s>", "<unk>", "<mask>"],
)
tokenizer.save_model(LOCAL_TOK_DIR)
print(f"Trained in {time.time()-t0:.0f}s")

for fname in ["vocab.json", "merges.txt"]:
    shutil.copy2(os.path.join(LOCAL_TOK_DIR, fname), os.path.join(DST_DATA_DIR, fname))
print("Saved to", DST_DATA_DIR)

## Phase 4 — Training

In [55]:
# Download facebook/bart-base weights, config, and tokenizer.
# Saves everything to Drive so subprocesses load offline via TRANSFORMERS_OFFLINE=1.
# safe_serialization=False forces pytorch_model.bin (SeqDiffuSeq doesn't support safetensors).
PRETRAINED_DIR = os.path.join(REPO_DIR, "pretrained", "bart-base")
CONFIG_NAME    = PRETRAINED_DIR

if os.path.exists(os.path.join(PRETRAINED_DIR, "pytorch_model.bin")):
    print("bart-base already cached →", PRETRAINED_DIR)
else:
    os.makedirs(PRETRAINED_DIR, exist_ok=True)
    from transformers import BartConfig, BartTokenizerFast, BartModel
    BartConfig.from_pretrained("facebook/bart-base").save_pretrained(PRETRAINED_DIR)
    BartTokenizerFast.from_pretrained("facebook/bart-base").save_pretrained(PRETRAINED_DIR)
    BartModel.from_pretrained("facebook/bart-base").save_pretrained(
        PRETRAINED_DIR, safe_serialization=False
    )
    print("bart-base saved to", PRETRAINED_DIR)
    print("Files:", sorted(os.listdir(PRETRAINED_DIR)))


bart-base already cached → /content/drive/MyDrive/BUyen_Qnhu++/src/SeqDiffuSeq/pretrained/bart-base


In [ ]:
import glob, re
os.makedirs(os.path.join(CKPT_DIR, "log"), exist_ok=True)

# Auto-resume: find latest non-EMA checkpoint in CKPT_DIR (on Drive — persists across sessions)
model_ckpts = sorted(c for c in glob.glob(os.path.join(CKPT_DIR, "model*.pt"))
                     if "ema" not in os.path.basename(c))
resume_ckpt = model_ckpts[-1] if model_ckpts else ""
if resume_ckpt:
    m = re.search(r"model0*(\d+)\.pt", os.path.basename(resume_ckpt))
    print(f"Resuming from step {int(m.group(1)) if m else '?'}: {os.path.basename(resume_ckpt)}")
else:
    print("No checkpoint found — starting from scratch.")

BATCH_SIZE = "64" if PLATFORM == "vastai" else "16"

args = [
VENV_PYTHON, "-u", "main.py",
"--checkpoint_path", CKPT_DIR,
"--src", "en",
"--tgt", "zh",
"--train_txt_path", "./data/en-zh/train",
"--val_txt_path", "./data/en-zh/valid",
"--dataset", "en-zh",
"--config_name", PRETRAINED,
"--diffusion_steps", "1000",
"--noise_schedule", "sqrt",
"--sequence_len", "64",
"--sequence_len_src", "128",
"--batch_size", BATCH_SIZE,
"--lr", "1e-4",
"--lr_anneal_steps", "100000",
"--resume_checkpoint", resume_ckpt,
"--warmup", "500",
"--save_interval", "5000",
"--eval_interval", "2500",
"--log_interval", "100",
"--schedule_update_stride", "200",
"--loss_update_granu", "20",
"--encoder_layers", "6",
"--decoder_layers", "6",
"--num_heads", "12",
"--in_channel", "768",
"--out_channel", "768",
"--num_channels", "3072",
"--vocab_size", "32005",
"--dropout", "0.3",
"--predict_xstart", "True",
"--seed", "42",
"--schedule_sampler", "uniform",
"--init_pretrained", "True",
"--freeze_embeddings", "False",
"--use_pretrained_embeddings", "False",
]

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"]  = "0"
env["DIFFUSION_BLOB_LOGDIR"] = os.path.join(CKPT_DIR, "log")
env["TRANSFORMERS_OFFLINE"]  = "1"

print(f"Batch size : {BATCH_SIZE}")
print("Starting training… logs →", os.path.join(CKPT_DIR, "log"))
result = subprocess.run(args, cwd=REPO_DIR, env=env, capture_output=True, text=True)
print(result.stdout[-3000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    print("Exit code:", result.returncode)

## Phase 5 — Inference

In [ ]:
import glob, re

# Find latest EMA checkpoint
ema_ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "ema_0.9999_*.pt")))
if not ema_ckpts:
    raise FileNotFoundError(f"No EMA checkpoint found in {CKPT_DIR}")
model_path = ema_ckpts[-1]
print(f"Checkpoint : {os.path.basename(model_path)}")

# Find latest time schedule (.npy saved by trainer every schedule_update_stride steps)
schedule_files = sorted(glob.glob(os.path.join(CKPT_DIR, "alpha_cumprod_step_*.npy")))
if not schedule_files:
    raise FileNotFoundError(f"No alpha_cumprod_step_*.npy found in {CKPT_DIR}")
schedule_path = schedule_files[-1]
print(f"Schedule   : {os.path.basename(schedule_path)}")

# Count test lines for verification (num_samples=-1 runs all)
test_src = os.path.join(REPO_DIR, "data", "en-zh", "test.en")
num_test  = sum(1 for _ in open(test_src, encoding="utf-8"))
print(f"Test lines : {num_test:,}  (num_samples=-1 → full set)")

inf_args = [
    VENV_PYTHON, "-u", "inference_main.py",
    "--model_name_or_path", model_path,
    "--val_txt_path",       "./data/en-zh/test",
    "--out_dir",            CKPT_DIR,
    "--time_schedule_path", schedule_path,
    "--diffusion_steps",    "200",
    "--num_samples",        "-1",
    "--batch_size",         "100",
    "--sequence_len",       "64",
    "--sequence_len_src",   "128",
    "--top_p",              "-1",
    "--clamp",              "no_clamp",
    "--use_ddim",           "True",
    "--seed",               "42",
    "--generate_by_q",      "False",
    "--generate_by_mix",    "False",
]

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"]  = "0"
env["TRANSFORMERS_OFFLINE"]  = "1"

print(f"\nStarting inference on {num_test:,} sentences…")
result = subprocess.run(inf_args, cwd=REPO_DIR, env=env, capture_output=True, text=True)
print(result.stdout[-3000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    print("Exit code:", result.returncode)


## Phase 6 — BLEU Evaluation

In [49]:
# Find the decoded output file (written to CKPT_DIR alongside the checkpoint)
decoded_files = sorted([
    f for f in glob.glob(os.path.join(CKPT_DIR, "ema_*.pt.samples_*.txt"))
    if "raw-output-ids" not in f
])

if not decoded_files:
    print("No decoded output file found in:", CKPT_DIR)
    print("Files present:")
    for f in glob.glob(os.path.join(CKPT_DIR, "*.txt")):
        print(" ", f)
else:
    output_file  = decoded_files[-1]
    # Short name: extract step number, e.g. "001000" -> "eval_step001000"
    import re
    m = re.search(r'ema_[\d.]+_(\d+)', os.path.basename(output_file))
    short = f"eval_step{m.group(1)}" if m else "eval"
    os.makedirs(OUT_DIR, exist_ok=True)
    eval_csv     = os.path.join(OUT_DIR, short + ".csv")
    eval_summary = os.path.join(OUT_DIR, short + "_summary.txt")
    print(f"Evaluating: {output_file}\n")

    eval_script = f"""
import json, csv, sacrebleu

with open({repr(output_file)}, "r", encoding="utf-8") as f:
    pairs = [json.loads(l.strip()) for l in f if l.strip()]

hypotheses = [p[0] for p in pairs]
references  = [p[1] for p in pairs]

src_path = {repr(os.path.join(REPO_DIR, "data", "en-zh", "test.en"))}
with open(src_path, "r", encoding="utf-8") as f:
    sources = [l.strip() for l in f if l.strip()]

n = min(len(sources), len(hypotheses))
sources, hypotheses, references = sources[:n], hypotheses[:n], references[:n]

bleu_char = sacrebleu.corpus_bleu(hypotheses, [references], tokenize='char')
bleu_13a  = sacrebleu.corpus_bleu(hypotheses, [references], tokenize='13a')

with open({repr(eval_csv)}, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["source_zh", "hypothesis_vi", "reference_vi"])
    for src, hyp, ref in zip(sources, hypotheses, references):
        writer.writerow([src, hyp, ref])

summary_text = "\\n".join([
    f"Output file     : {repr(output_file)}",
    f"CSV             : {repr(eval_csv)}",
    f"Num samples     : {{n}}",
    "",
    f"SacreBLEU (char): {{bleu_char.score:.2f}}",
    f"SacreBLEU (13a) : {{bleu_13a.score:.2f}}",
])
print(summary_text)
with open({repr(eval_summary)}, "w", encoding="utf-8") as f:
    f.write(summary_text)
print(f"\\nCSV     saved to: {repr(eval_csv)}")
print(f"Summary saved to: {repr(eval_summary)}")
"""

    result = subprocess.run(
        [VENV_PYTHON, "-c", eval_script],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:])


Evaluating: /content/drive/MyDrive/BUyen_Qnhu++/src/SeqDiffuSeq/ckpts/en_zh/ema_0.9999_030000.pt.samples_50.steps-1000.clamp-clamp-normal_101.txt

Output file     : '/content/drive/MyDrive/BUyen_Qnhu++/src/SeqDiffuSeq/ckpts/en_zh/ema_0.9999_030000.pt.samples_50.steps-1000.clamp-clamp-normal_101.txt'
CSV             : '/content/drive/MyDrive/BUyen_Qnhu++/src/SeqDiffuSeq/ckpts/en_zh/inference_out/eval_step030000.csv'
Num samples     : 50

SacreBLEU (char): 0.23
SacreBLEU (13a) : 0.04

CSV     saved to: '/content/drive/MyDrive/BUyen_Qnhu++/src/SeqDiffuSeq/ckpts/en_zh/inference_out/eval_step030000.csv'
Summary saved to: '/content/drive/MyDrive/BUyen_Qnhu++/src/SeqDiffuSeq/ckpts/en_zh/inference_out/eval_step030000_summary.txt'

